# Decoder 训练、Alpha-Lambda 消融与闭环 L2

这个 notebook 是 CartPole 和 Pendulum decoder 实验的统一交互入口，支持：

1. 单独训练 intensity / saliency decoder；
2. 用固定 `seed=2025` 运行完整 `7 × 7` 的 `alpha × lambda_ctrl` 消融；
3. 为 canonical baseline 和全部网格 checkpoint 生成 DWM-only rollout；
4. 汇总单帧指标与 full-state L2 表；
5. 在查看 validation 对照后，显式晋升选中的 saliency 配置。

本轮不运行 StarV，也不生成 `transition_dataset.npz`。昂贵 cell 默认注释，按章节手工执行即可断点续跑。

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    candidate
    for candidate in (cwd, cwd.parent)
    if (candidate / "train_decoder.py").exists()
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import ablation as abl
import train_decoder as td
from utils import load_config, resolve_device, set_seed

print(f"repo={REPO_ROOT}")

## 1. 单模型训练

`run_train` 继续复用 `train_decoder.train`。默认 JSON 就是 canonical 配置；传入 override 可以临时训练其他参数，但完整消融请使用下一节的网格编排。

In [ ]:
def run_train(env, mode="saliency", *, alpha=None, lambda_ctrl=None, seed=None, output_dir=None):
    config_path = Path("config/train_decoder") / env / f"{mode}.json"
    config = load_config(config_path)
    if alpha is not None:
        config["weight"]["alpha"] = float(alpha)
    if lambda_ctrl is not None:
        config["lambda_ctrl"] = float(lambda_ctrl)
    if seed is not None:
        config["training"]["seed"] = int(seed)
    if output_dir is not None:
        config["output_dir"] = str(output_dir)

    set_seed(config["training"]["seed"])
    device = resolve_device(config.get("device", "auto"))
    td.train(config, device)
    return Path(config["output_dir"]) / "metrics.json"

# 示例（会实际训练，按需取消注释）：
# run_train("pendulum", "saliency")
# run_train("cartpole", "intensity")

## 2. 完整 Alpha-Lambda 网格

每个环境固定 49 点，固定 `seed=2025`，不筛 seed。目录为：

`dwm_weight/now_weight/<env>/alpha_lambda_grid/alpha_<a>/lambda_<λ>/seed_2025/`

完整且参数匹配的点会跳过；部分结果或参数不一致的点会重跑。

In [ ]:
ENVS = ("pendulum", "cartpole")
ALPHAS = abl.DEFAULT_ALPHAS
LAMBDAS = abl.DEFAULT_LAMBDAS
SEED = abl.DEFAULT_SEED

EXPERIMENTS = {
    env: abl.build_experiments(env, ALPHAS, LAMBDAS, SEED)
    for env in ENVS
}
for env, experiments in EXPERIMENTS.items():
    print(env, len(experiments), experiments[0].output_dir)

In [ ]:
# 昂贵操作：两个环境共 98 次训练。取消注释后可断点续跑。
# training_status = {}
# for env in ENVS:
#     frame = abl.run_training_grid(env, skip_existing=True, continue_on_error=True)
#     training_status[env] = frame
#     display(frame["status"].value_counts())
#     failures = frame.loc[frame["status"] == "failed"]
#     if not failures.empty:
#         display(failures)

## 3. 单帧训练指标

从每个网格点的 `metrics.json` 读取 best epoch 的 validation 指标和最终 test 指标。validation 用来选择超参数；test 不参与调参。

In [ ]:
training_tables = {}
for env in ENVS:
    try:
        training_tables[env] = abl.collect_training_metrics(EXPERIMENTS[env])
        print(f"[{env}] rows={len(training_tables[env])}")
        display(
            training_tables[env]
            .loc[lambda frame: frame["split"] == "val"]
            .sort_values(["ctrl_mse", "pixel_mse"])
            .head(10)
        )
    except FileNotFoundError as exc:
        print(f"[{env}] 网格尚未完整：{exc}")

## 4. Canonical baseline 的 DWM-only rollout

重新生成 intensity / saliency 闭环 trajectory。该流程复用共享的 `starv_states.npz`，不渲染真实图像，也不生成单步 transition 数据。

In [ ]:
# 昂贵操作：按需取消注释。已有且溯源匹配的结果会跳过。
# mainline_rollouts = {}
# for env in ENVS:
#     mainline_rollouts[env] = abl.run_mainline_rollouts(
#         env,
#         variants=("intensity", "saliency"),
#         skip_existing=True,
#     )
#     display(mainline_rollouts[env])

## 5. 完整网格的 DWM-only rollout

每个成功训练的点都进入 rollout，不按单帧指标提前筛选。variant、checkpoint 和三个 split 的初始状态都通过验证后才允许断点跳过。

In [ ]:
# 昂贵操作：两个环境共 98 次 DWM-only rollout。取消注释后可断点续跑。
# rollout_status = {}
# for env in ENVS:
#     frame = abl.run_rollout_grid(env, skip_existing=True, continue_on_error=True)
#     rollout_status[env] = frame
#     display(frame["status"].value_counts())
#     failures = frame.loc[frame["status"] == "failed"]
#     if not failures.empty:
#         display(failures)

## 6. 闭环 L2 汇总

CartPole 使用 full-state L2。Pendulum 对 θ 使用最短圆周差后再计算 full-state L2。每个环境必须有 49 个完整网格点，才会写出 98 行（val/test）的 `combined_metrics.csv`。

In [ ]:
tables = {}
for env in ENVS:
    try:
        tables[env] = abl.write_summary_tables(env)
        combined = tables[env]["combined"]
        print(f"[{env}] combined rows={len(combined)}")
        display(combined)
        print(tables[env]["paths"])
    except FileNotFoundError as exc:
        print(f"[{env}] 训练或 rollout 尚未完整：{exc}")

## 7. Validation pivot 表

优先查看闭环 `max_l2_p95` / `max_l2_mean`，并同时核对 controller MSE 与 pixel MSE。test 只在候选敲定后查看。

In [ ]:
for env, result in tables.items():
    combined = result["combined"]
    for metric in ("max_l2_p95", "max_l2_mean", "ctrl_mse", "pixel_mse"):
        print(f"[{env}] validation {metric}")
        display(abl.pivot_metric(combined, metric, split="val"))

## 8. 显式晋升 saliency mainline

先显示候选与当前 mainline 的 val/test 单帧和 L2 对照。确认后才执行 `force=True`：它会用选中参数重训 canonical saliency 并重生成 canonical rollout。

如果候选仍是 `(alpha=8, lambda_ctrl=0.1)`，训练 JSON 不会改写；无论候选是什么，sampling 和 StarV JSON 的 canonical checkpoint 路径都保持不变。

In [ ]:
# 必须手工填写 validation 表中确认的组合，再逐行取消注释。
# 7/13 validation max_l2_p95 第一名是 Pendulum (alpha=16, lambda_ctrl=0.5)。
# candidate = abl.Experiment("pendulum", alpha=16.0, lambda_ctrl=0.5, seed=SEED)
# display(abl.compare_with_mainline(candidate))
# promotion = abl.promote_mainline(candidate, force=True)
# display(promotion["rollout"])
#
# 晋升后重新生成汇总并查看最终 baseline 对照：
# display(abl.compare_with_mainline(candidate))

## 运行顺序与边界

推荐顺序：网格训练 → 单帧表 → mainline rollout → 网格 rollout → L2/CSV → validation 候选对照 → 人工晋升。

本 notebook 不运行 StarV，不做 conformal inflation，也不恢复 `transition_dataset.npz`。旧的 CartPole 一维 alpha/lambda 目录保留为历史记录，新二维网格使用独立目录。